# try-except-solve — worked example 3: Solve with a default fallback vector

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `try-except-solve`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Sometimes the caller wants a concrete fallback rather than `None`. The same `try/except RuntimeError` wraps the solve, but the except branch returns a caller-supplied default tensor (e.g. zeros) so downstream code always receives a same-shaped result.

## Worked solution

We solve `Ax = b`, but on failure we return a supplied default vector instead of `None`.

1. The `try` block attempts `t.linalg.solve(A, b)` and returns it on success.
2. On `RuntimeError` (singular `A`), we return the `default` tensor, guaranteeing the caller always gets a vector of the expected shape.
3. This is useful in pipelines that cannot tolerate `None` — the default acts as a neutral placeholder.

We show an invertible solve returning the true answer and a singular solve returning the zeros default.

In [ ]:
import torch as t

def solve_or_default(A: t.Tensor, b: t.Tensor, default: t.Tensor) -> t.Tensor:
    try:
        return t.linalg.solve(A, b)
    except RuntimeError:
        return default

A_good = t.tensor([[5.0, 0.0], [0.0, 2.0]])
b = t.tensor([10.0, 6.0])
A_sing = t.tensor([[0.0, 0.0], [0.0, 0.0]])
fallback = t.zeros(2)
print('good:', solve_or_default(A_good, b, fallback).tolist())
print('singular:', solve_or_default(A_sing, b, fallback).tolist())